# AXI-Lite 寄存器独立测试
先使用 board_test.py 完成停止视频服务、板测与恢复。交互操作前应确保当前加载的是匹配的测试 Overlay，且只有一个操作者。下列单元格不会自动下载位流。

In [ ]:
from pathlib import Path
import fcntl
import hashlib
import json
from contextlib import ExitStack
from pynq import Overlay
from axilt import Axilt
bit = Path('/home/xilinx/axilt_test/AXI_LITE_test.bit')
manifest = json.loads(bit.with_name('artifact_manifest.json').read_text())
for suffix in ('.bit', '.hwh'):
    path = bit.with_suffix(suffix)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == manifest['files'][path.name]['sha256']
overlay = Overlay(str(bit), download=False)
assert overlay.is_loaded(), '请先加载并核对独立测试 Overlay'
print('元数据已核对，下面单元格在独占锁内执行测试。')

In [ ]:
with ExitStack() as locks:
    for name in ('ees331_axilt', 'ees331_camera'):
        lock = locks.enter_context(open('/run/lock/' + name + '.lock', 'a'))
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    assert overlay.is_loaded()
    driver = Axilt.from_overlay(overlay, './libmmio_ordered.so')
    driver.write('SCRATCH', 0x12345678)
    print(hex(driver.read('SCRATCH')))
    print(driver.execute(0x12345678))  # 预期结果 0xEDCBA987


交互单次演示不作为正式板测PASS。请在测试结束后恢复原视频服务，并检查HDMI与UDP新帧。
